# 09b — PRIO-GRID Feature Engineering Methods

**Depends on:** notebook 09 (PRIO-GRID pull), notebook 04 (UCDP-GED events)  
**Reads from ADLS:** `raw/prio_grid/`, `raw/ucdp_ged/`  
**Writes to ADLS:** `raw/prio_grid/{RUN_DATE}/priogrid_engineered_features.parquet`  

## Purpose

Notebook 09 produces a simple country-year aggregation (mean/sum/max per variable).
This notebook builds eight additional feature families from the same grid-cell data
that are more predictive of instability outcomes because they capture *within-country
variation* — the distributional shape, spatial concentration, and temporal shocks
that flat country-level averages erase.

All outputs are at **country-year** resolution and are ready to join into the
feature matrix built in notebook 14.

## Eight methods

| # | Method | Instability mechanism |
|---|---|---|
| 1 | **Distributional aggregates** | Dispersion of NTL/pop captures urban–rural fractures |
| 2 | **NTL Gini / darkness fraction** | Within-country economic inequality proxy |
| 3 | **Population-weighted climate stress** | Which populations face rainfall/temp volatility |
| 4 | **Conflict cell density (UCDP-GED join)** | Spatial breadth of active violence |
| 5 | **Lootable resource population exposure** | Grievance potential of resource enclaves |
| 6 | **Economic polarization (core vs. periphery)** | Primate city concentration, regional neglect |
| 7 | **NTL economic shock index** | Sudden economic deterioration signal |
| 8 | **GeoEPR ethnic territory join** | Economic conditions inside excluded group homelands |

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER  (default: 'data')
```

In [ ]:
import os
import re
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from azure.identity import DefaultAzureCredential
import adlfs

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

PANEL_START_YEAR = 2000
PANEL_END_YEAR   = 2024

# NTL thresholds (DMSP-OLS calibrated scale: 0–63)
NTL_DARK_THRESHOLD   = 3    # below this = essentially unlit
NTL_URBAN_THRESHOLD  = 20   # above this = urban / semi-urban

# Climate stress: top-quartile rainfall SD within country-year is "high stress"
CLIMATE_STRESS_QUANTILE = 0.75

# NTL shock: flag if pop-weighted NTL drops more than this many SDs below 5-yr baseline
NTL_SHOCK_THRESHOLD = 1.5

print(f"Run date     : {RUN_DATE}")
print(f"Panel years  : {PANEL_START_YEAR}–{PANEL_END_YEAR}")

## ADLS helpers

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

def read_latest_parquet(prefix: str, filename_hint: str = "") -> pd.DataFrame | None:
    """Read the parquet from the lexicographically latest date partition under prefix."""
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    full_prefix = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = fs.ls(full_prefix, detail=False)
    except FileNotFoundError:
        print(f"  WARNING: {full_prefix} not found")
        return None

    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)],
        reverse=True,
    )
    if not date_dirs:
        print(f"  WARNING: no date partitions under {full_prefix}")
        return None

    pattern = f"{date_dirs[0]}/*{filename_hint}*.parquet" if filename_hint else f"{date_dirs[0]}/*.parquet"
    files = [
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + f.replace(f"{ADLS_CONTAINER}/", "", 1)
        for f in fs.glob(pattern)
    ]
    if not files:
        print(f"  WARNING: no parquet files at {date_dirs[0]}")
        return None

    dfs = [pd.read_parquet(p, storage_options=storage_options) for p in files]
    df  = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
    print(f"  Loaded {len(df):,} rows ← {date_dirs[0]}")
    return df

## Load PRIO-GRID data

We need two tables produced by notebook 09:
- **Static** — one row per grid cell with geographic invariants (`bdist1`, `mountains_mean`, etc.)
- **Yearly** — one row per (cell, year) with time-varying variables (`pop_gpw_sum`, `nlights_calib_mean`, etc.)

And one table from notebook 04:
- **UCDP-GED events** — one row per conflict event with a `priogrid_gid` field, used in Method 4.

In [ ]:
# ── PRIO-GRID static ──────────────────────────────────────────────────────────
df_static = read_latest_parquet("raw/prio_grid", filename_hint="static")

# ── PRIO-GRID yearly ─────────────────────────────────────────────────────────
df_yearly = read_latest_parquet("raw/prio_grid", filename_hint="yearly")

# ── UCDP-GED events (for Method 4) ───────────────────────────────────────────
df_ged = read_latest_parquet("raw/ucdp_ged", filename_hint="events")

for name, df in [("static", df_static), ("yearly", df_yearly), ("ged", df_ged)]:
    if df is not None:
        df.columns = [c.lower().strip() for c in df.columns]
        print(f"  {name:8s}: {df.shape}  cols={list(df.columns[:8])}...")
    else:
        print(f"  {name:8s}: NOT LOADED — some methods will be skipped")

## Country crosswalk (GW code → ISO3)

PRIO-GRID uses Gleditsch-Ward (`gwno`) country codes. We map these to ISO3 for joining
to the country-year feature matrix.

In [ ]:
_cw_path = next(
    (p for p in [Path("../data/country_crosswalk.csv"), Path("data/country_crosswalk.csv")]
     if p.exists()),
    None,
)
if _cw_path:
    df_cw = pd.read_csv(_cw_path, dtype=str)
    df_cw["gw_numeric"] = pd.to_numeric(df_cw["gw_numeric"], errors="coerce")
    gw_to_iso3 = dict(zip(df_cw["gw_numeric"], df_cw["iso3"]))
    print(f"Crosswalk loaded: {len(df_cw)} countries")
else:
    print("WARNING: country_crosswalk.csv not found — gwno→iso3 mapping unavailable")
    gw_to_iso3 = {}

## Merge static variables into yearly panel

Joins geographic invariants (border distance, terrain, forest cover) onto the yearly
time-varying rows so that every cell-year has both geographic and temporal attributes.
Also maps `gwno` → `iso3` and filters to the panel years.

In [ ]:
df_panel = None

if df_yearly is not None:
    df_panel = df_yearly.copy()

    # Merge static geography onto yearly rows
    if df_static is not None:
        static_cols = [c for c in df_static.columns
                       if c not in df_panel.columns or c == "gid"]
        df_panel = df_panel.merge(df_static[static_cols], on="gid", how="left")

    # Resolve country code: prefer gwno column
    gwno_col = next((c for c in df_panel.columns if c in ("gwno", "gwnoa", "gwno_a")), None)
    if gwno_col and gw_to_iso3:
        df_panel["iso3"] = pd.to_numeric(df_panel[gwno_col], errors="coerce").map(gw_to_iso3)

    # Ensure numeric types
    df_panel["year"] = pd.to_numeric(df_panel["year"], errors="coerce").astype("Int64")
    for col in ["pop_gpw_sum", "nlights_calib_mean", "rainfall_sd", "temp_sd", "ttime_mean"]:
        if col in df_panel.columns:
            df_panel[col] = pd.to_numeric(df_panel[col], errors="coerce")

    # Filter to panel years
    df_panel = df_panel[
        df_panel["year"].between(PANEL_START_YEAR, PANEL_END_YEAR)
    ].copy()

    print(f"Cell-year panel : {len(df_panel):,} rows")
    print(f"Years           : {df_panel['year'].min()}–{df_panel['year'].max()}")
    print(f"Cells           : {df_panel['gid'].nunique():,}")
    print(f"Countries (iso3): {df_panel['iso3'].nunique() if 'iso3' in df_panel.columns else 'n/a'}")
else:
    print("WARNING: yearly panel not loaded — cannot proceed with feature engineering")

## Method 1 — Enhanced distributional aggregates

**What it adds:** Standard notebook 09 aggregates NTL and population to country-year means,
sums, and max values. Those statistics describe the *level* but miss the *shape* of the
distribution within each country. A country where 5% of cells hold 90% of the light is
structurally different from one with evenly distributed light — even if their means are
identical.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_ntl_mean` | Mean nighttime light intensity across cells |
| `prio_ntl_std` | Std dev of NTL — within-country economic dispersion |
| `prio_ntl_p10` | 10th-percentile NTL (how dark are the dark areas?) |
| `prio_ntl_p90` | 90th-percentile NTL (how bright are the bright areas?) |
| `prio_ntl_p90p10_ratio` | p90 / p10 — distributional spread (high = stark inequality) |
| `prio_pop_sum` | Total country population (sum of cell populations) |
| `prio_pop_std` | Std dev of cell-level population — spatial concentration of people |
| `prio_pop_top10cell_share` | Fraction of population in the 10% most populated cells |

**Why it matters for instability models:**
High `prio_ntl_p90p10_ratio` and `prio_pop_top10cell_share` both proxy for the
spatial concentration of wealth and people that Cederman et al. (2011) link to
horizontal inequality and conflict risk.

In [ ]:
df_m1 = pd.DataFrame()

if df_panel is not None and "iso3" in df_panel.columns:
    ntl_col = "nlights_calib_mean"
    pop_col = "pop_gpw_sum"

    grp = df_panel.groupby(["iso3", "year"])

    agg = pd.DataFrame()

    if ntl_col in df_panel.columns:
        ntl = grp[ntl_col]
        agg["prio_ntl_mean"]        = ntl.mean()
        agg["prio_ntl_std"]         = ntl.std()
        agg["prio_ntl_p10"]         = ntl.quantile(0.10)
        agg["prio_ntl_p90"]         = ntl.quantile(0.90)
        p10 = agg["prio_ntl_p10"].replace(0, np.nan)
        agg["prio_ntl_p90p10_ratio"] = agg["prio_ntl_p90"] / p10

    if pop_col in df_panel.columns:
        pop = grp[pop_col]
        agg["prio_pop_sum"] = pop.sum()
        agg["prio_pop_std"] = pop.std()

        # Fraction of population in the top-10% most populated cells
        def _top10_share(s):
            s = s.dropna()
            if s.sum() == 0 or len(s) == 0:
                return np.nan
            threshold = s.quantile(0.90)
            return s[s >= threshold].sum() / s.sum()

        agg["prio_pop_top10cell_share"] = grp[pop_col].apply(_top10_share)

    df_m1 = agg.reset_index()
    print(f"Method 1: {len(df_m1):,} rows × {len(df_m1.columns)} columns")
    print(df_m1.describe().round(3))
else:
    print("Method 1 skipped — panel not available")

## Method 2 — NTL Gini coefficient and darkness fraction

**What it adds:** The Gini coefficient of nighttime light values across a country's grid
cells is a direct spatial proxy for economic inequality within a country. A Gini of 0
means every cell is equally lit; a Gini near 1 means nearly all light is concentrated
in a handful of cells. This is distinct from aggregate GDP inequality measures because
it captures the geographic dimension — whether inequality is expressed as spatial
separation between rich and poor regions.

The darkness fraction captures a complementary dimension: what share of the country's
area is in effective darkness, regardless of the distribution among lit areas.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_ntl_gini` | Gini coefficient of NTL across cells (0 = equal, 1 = concentrated) |
| `prio_ntl_darkness_frac` | Fraction of cells with NTL < 3 (essentially unlit) |
| `prio_ntl_urban_frac` | Fraction of cells with NTL > 20 (urban / semi-urban proxy) |

**Why it matters:** Høyland et al. (2012) and Lessmann & Seidel (2017) show that
within-country spatial inequality (proxied by NTL Gini) is a robust predictor of
separatist conflict and ethnic civil war onset, over and above national-level
inequality measures like the Gini of household income.

In [ ]:
def _gini(values: np.ndarray) -> float:
    """Gini coefficient for a 1-D array of non-negative values."""
    v = values[~np.isnan(values)]
    v = v[v >= 0]
    if len(v) == 0 or v.sum() == 0:
        return np.nan
    v = np.sort(v)
    n = len(v)
    idx = np.arange(1, n + 1)
    return float((2 * (idx * v).sum()) / (n * v.sum()) - (n + 1) / n)


df_m2 = pd.DataFrame()

if df_panel is not None and "iso3" in df_panel.columns:
    ntl_col = "nlights_calib_mean"

    if ntl_col not in df_panel.columns:
        print(f"Method 2 skipped — '{ntl_col}' not in panel")
    else:
        grp = df_panel.groupby(["iso3", "year"])[ntl_col]

        ntl_gini = grp.apply(lambda s: _gini(s.values)).rename("prio_ntl_gini")

        ntl_darkness = grp.apply(
            lambda s: (s < NTL_DARK_THRESHOLD).sum() / max(len(s), 1)
        ).rename("prio_ntl_darkness_frac")

        ntl_urban = grp.apply(
            lambda s: (s > NTL_URBAN_THRESHOLD).sum() / max(len(s), 1)
        ).rename("prio_ntl_urban_frac")

        df_m2 = pd.concat([ntl_gini, ntl_darkness, ntl_urban], axis=1).reset_index()

        print(f"Method 2: {len(df_m2):,} rows × {len(df_m2.columns)} columns")
        print(df_m2[["prio_ntl_gini", "prio_ntl_darkness_frac", "prio_ntl_urban_frac"]].describe().round(3))
else:
    print("Method 2 skipped — panel not available")

## Method 3 — Population-weighted climate stress

**What it adds:** Simple country-year means of rainfall variability (`rainfall_sd`) and
temperature variability (`temp_sd`) treat all cells equally. But a drought that hits
an empty desert is not the same as one that hits a densely populated agricultural
region. Population-weighting shifts the measure from geographic exposure to *human*
exposure — the fraction of the population actually experiencing climate volatility.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_pop_wtd_rainfall_sd` | Population-weighted mean rainfall SD across cells |
| `prio_pop_wtd_temp_sd` | Population-weighted mean temperature SD across cells |
| `prio_high_stress_pop_frac` | Fraction of population in cells above the country's 75th-percentile rainfall SD |

**Why it matters:** Burke et al. (2015) and Hsiang et al. (2013) establish a causal
link between climate shocks and conflict, but the effect is concentrated where people
live. `prio_high_stress_pop_frac` directly captures what share of the population is
exposed to the worst climate volatility in their country — a more precise grievance
signal than a national average.

In [ ]:
def _pop_weighted_mean(df_grp, value_col, weight_col):
    """Population-weighted mean of value_col within each (iso3, year) group."""
    def _pwm(g):
        v = g[value_col].values.astype(float)
        w = g[weight_col].values.astype(float)
        mask = ~(np.isnan(v) | np.isnan(w)) & (w > 0)
        if mask.sum() == 0:
            return np.nan
        return np.average(v[mask], weights=w[mask])
    return df_grp.apply(_pwm)


df_m3 = pd.DataFrame()

if df_panel is not None and "iso3" in df_panel.columns:
    pop_col = "pop_gpw_sum"
    rain_col = "rainfall_sd"
    temp_col = "temp_sd"

    missing = [c for c in [pop_col, rain_col] if c not in df_panel.columns]
    if missing:
        print(f"Method 3 skipped — missing columns: {missing}")
    else:
        grp = df_panel.groupby(["iso3", "year"])

        rows = {}

        if rain_col in df_panel.columns:
            rows["prio_pop_wtd_rainfall_sd"] = _pop_weighted_mean(grp, rain_col, pop_col)

        if temp_col in df_panel.columns:
            rows["prio_pop_wtd_temp_sd"] = _pop_weighted_mean(grp, temp_col, pop_col)

        # Fraction of population in high-stress cells (above country 75th-pct rainfall SD)
        def _high_stress_pop_frac(g):
            if rain_col not in g.columns or pop_col not in g.columns:
                return np.nan
            r = g[rain_col].values.astype(float)
            p = g[pop_col].fillna(0).values.astype(float)
            if p.sum() == 0 or np.isnan(r).all():
                return np.nan
            threshold = np.nanquantile(r, CLIMATE_STRESS_QUANTILE)
            return p[r >= threshold].sum() / p.sum()

        rows["prio_high_stress_pop_frac"] = grp.apply(_high_stress_pop_frac)

        df_m3 = pd.DataFrame(rows).reset_index()
        print(f"Method 3: {len(df_m3):,} rows × {len(df_m3.columns)} columns")
        print(df_m3.drop(columns=["iso3", "year"]).describe().round(3))
else:
    print("Method 3 skipped — panel not available")

## Method 4 — Conflict cell density (UCDP-GED × PRIO-GRID join)

**What it adds:** UCDP-GED event records each include a `priogrid_gid` field — the
0.5° cell where the event occurred. Joining events back to the grid lets us compute
the *spatial footprint* of conflict within each country-year: how many cells had any
violence, what fraction of the country's area that represents, and what fraction of
the population lived inside conflict-affected cells.

This distinguishes spatially concentrated conflicts (insurgent enclaves, border wars)
from diffuse ones (nationwide breakdown), which have different escalation dynamics and
different policy implications.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_conflict_cell_count` | Number of distinct cells with ≥1 state-based violence event |
| `prio_conflict_cell_frac` | Conflict cells / total country cells — spatial footprint |
| `prio_conflict_pop_frac` | Population in conflict cells / total country population |
| `prio_conflict_deaths_per_cell` | Battle deaths (best estimate) per active conflict cell |

**Why it matters:** `prio_conflict_cell_frac` distinguishes a country where violence
is confined to one border region from one in nationwide collapse. `prio_conflict_pop_frac`
captures how many civilians are in the conflict zone — a direct input to humanitarian
crisis escalation models.

In [ ]:
df_m4 = pd.DataFrame()

if df_panel is not None and df_ged is not None and "iso3" in df_panel.columns:
    # State-based violence only (type_of_violence == 1)
    ged_sb = df_ged[df_ged.get("type_of_violence", pd.Series(dtype=int)) == 1].copy() \
        if "type_of_violence" in df_ged.columns \
        else df_ged.copy()

    gid_col  = next((c for c in ged_sb.columns if c in ("priogrid_gid", "priogrid_id", "gid")), None)
    yr_col   = next((c for c in ged_sb.columns if c == "year"), None)
    best_col = next((c for c in ged_sb.columns if c == "best"), None)

    if not gid_col or not yr_col:
        print(f"Method 4 skipped — UCDP-GED missing gid/year columns (found: {list(ged_sb.columns[:10])})")
    else:
        ged_sb[yr_col]  = pd.to_numeric(ged_sb[yr_col],  errors="coerce").astype("Int64")
        ged_sb[gid_col] = pd.to_numeric(ged_sb[gid_col], errors="coerce")

        # Aggregate GED to cell-year: distinct event count + total deaths
        ged_agg = (
            ged_sb.groupby([gid_col, yr_col], as_index=False)
            .agg(
                event_count=(gid_col, "count"),
                **({best_col: (best_col, "sum")} if best_col else {}),
            )
            .rename(columns={gid_col: "gid", yr_col: "year"})
        )

        # Join cell-level population and iso3 from panel (one row per cell-year)
        cell_info = df_panel[["gid", "year", "iso3", "pop_gpw_sum"]].drop_duplicates()
        ged_cells = ged_agg.merge(cell_info, on=["gid", "year"], how="left")

        # Total cells per country-year (denominator)
        total_cells = (
            cell_info.groupby(["iso3", "year"])["gid"]
            .nunique()
            .rename("total_cells")
            .reset_index()
        )

        # Total population per country-year
        total_pop = (
            cell_info.groupby(["iso3", "year"])["pop_gpw_sum"]
            .sum()
            .rename("total_pop")
            .reset_index()
        )

        # Conflict-cell aggregates per country-year
        ged_cy = (
            ged_cells.groupby(["iso3", "year"], as_index=False)
            .agg(
                prio_conflict_cell_count=("gid", "nunique"),
                prio_conflict_pop=("pop_gpw_sum", "sum"),
                **({best_col: (best_col, "sum")} if best_col in ged_cells.columns else {}),
            )
        )

        df_m4 = (
            ged_cy
            .merge(total_cells, on=["iso3", "year"], how="left")
            .merge(total_pop,   on=["iso3", "year"], how="left")
        )

        df_m4["prio_conflict_cell_frac"] = (
            df_m4["prio_conflict_cell_count"] / df_m4["total_cells"].replace(0, np.nan)
        )
        df_m4["prio_conflict_pop_frac"] = (
            df_m4["prio_conflict_pop"] / df_m4["total_pop"].replace(0, np.nan)
        )
        if best_col in df_m4.columns:
            df_m4["prio_conflict_deaths_per_cell"] = (
                df_m4[best_col] / df_m4["prio_conflict_cell_count"].replace(0, np.nan)
            )
            df_m4 = df_m4.drop(columns=[best_col])

        df_m4 = df_m4.drop(columns=["prio_conflict_pop", "total_cells", "total_pop"])

        print(f"Method 4: {len(df_m4):,} rows × {len(df_m4.columns)} columns")
        print(df_m4.drop(columns=["iso3", "year"]).describe().round(3))
else:
    print("Method 4 skipped — panel or UCDP-GED events not available")

## Method 5 — Lootable resource population exposure

**What it adds:** PRIO-GRID flags each cell for petroleum extraction (`petroleum_y`)
and drug crop cultivation (`drug_y`). Counting these cells at country level tells you
whether resources exist; weighting by population tells you whether those resources are
near concentrations of people who could contest or benefit from them.

Resource enclaves near large populations create more acute grievance dynamics than
remote extraction. Population exposure is the relevant variable for models predicting
unrest onset, not mere resource presence.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_petroleum_cell_frac` | Fraction of country cells with petroleum extraction |
| `prio_petroleum_pop_frac` | Fraction of country population in petroleum cells |
| `prio_drug_cell_frac` | Fraction of country cells with drug crop cultivation |
| `prio_drug_pop_frac` | Fraction of country population in drug cultivation cells |
| `prio_resource_any_pop_frac` | Fraction of population in any resource cell (petroleum or drug) |

**Why it matters:** Ross (2004) and Lujala (2010) show that lootable resources
(especially gemstones and drugs) proximate to population centers are more strongly
associated with conflict onset than remote point-source resources. These features
operationalize that spatial proximity directly.

In [ ]:
df_m5 = pd.DataFrame()

if df_panel is not None and "iso3" in df_panel.columns:
    pop_col = "pop_gpw_sum"
    pet_col = "petroleum_y"
    drug_col = "drug_y"

    available = [c for c in [pet_col, drug_col] if c in df_panel.columns]
    if not available or pop_col not in df_panel.columns:
        print(f"Method 5 skipped — missing: {[c for c in [pop_col, pet_col, drug_col] if c not in df_panel.columns]}")
    else:
        grp = df_panel.groupby(["iso3", "year"])
        total_cells = grp["gid"].nunique().rename("total_cells")
        total_pop   = grp[pop_col].sum().rename("total_pop")

        rows = {}

        for flag_col, prefix in [(pet_col, "petroleum"), (drug_col, "drug")]:
            if flag_col not in df_panel.columns:
                continue

            flag = df_panel[flag_col].fillna(0).astype(float)
            flag_mask = df_panel.copy()
            flag_mask["_flag"] = (flag > 0).astype(float)
            flag_mask["_flag_pop"] = flag_mask["_flag"] * flag_mask[pop_col].fillna(0)

            fg = flag_mask.groupby(["iso3", "year"])
            rows[f"prio_{prefix}_cell_frac"] = (
                fg["_flag"].sum() / total_cells.replace(0, np.nan)
            )
            rows[f"prio_{prefix}_pop_frac"] = (
                fg["_flag_pop"].sum() / total_pop.replace(0, np.nan)
            )

        # Any resource
        if pet_col in df_panel.columns and drug_col in df_panel.columns:
            df_panel["_any_resource"] = (
                (df_panel[pet_col].fillna(0) > 0) | (df_panel[drug_col].fillna(0) > 0)
            ).astype(float)
            df_panel["_any_resource_pop"] = df_panel["_any_resource"] * df_panel[pop_col].fillna(0)
            ag = df_panel.groupby(["iso3", "year"])
            rows["prio_resource_any_pop_frac"] = (
                ag["_any_resource_pop"].sum() / total_pop.replace(0, np.nan)
            )
            df_panel.drop(columns=["_any_resource", "_any_resource_pop"], inplace=True)

        df_m5 = pd.DataFrame(rows).reset_index()
        print(f"Method 5: {len(df_m5):,} rows × {len(df_m5.columns)} columns")
        print(df_m5.drop(columns=["iso3", "year"]).describe().round(3))
else:
    print("Method 5 skipped — panel not available")

## Method 6 — Economic polarization (core vs. periphery)

**What it adds:** In many fragile states, economic activity is concentrated in a
primate city while the rest of the country is in darkness. This core–periphery
structure is a spatial expression of the centre–periphery grievance dynamic that
drives secessionist conflict and rural insurgency.

We approximate the "economic core" as the top decile of cells by NTL within a
country-year, and the "periphery" as the bottom half. The ratio of their mean NTL
values captures how much brighter the core is relative to the periphery — a direct
proxy for spatial economic polarization.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_ntl_core_mean` | Mean NTL of top-10% brightest cells (economic core proxy) |
| `prio_ntl_periphery_mean` | Mean NTL of bottom-50% cells (periphery) |
| `prio_ntl_core_periphery_ratio` | Core mean / periphery mean — polarization index |
| `prio_ntl_cv` | Coefficient of variation of NTL (std/mean) — normalized dispersion |

**Why it matters:** High `prio_ntl_core_periphery_ratio` captures what Bakke &
Wibbels (2006) call "regional economic inequality" — a predictor of both ethnic
separatism and coup risk, because peripheral elites have both grievance and
organizational capacity to challenge the centre.

In [ ]:
df_m6 = pd.DataFrame()

if df_panel is not None and "iso3" in df_panel.columns:
    ntl_col = "nlights_calib_mean"

    if ntl_col not in df_panel.columns:
        print(f"Method 6 skipped — '{ntl_col}' not in panel")
    else:
        def _polarization(s):
            v = s.dropna().values
            if len(v) < 4:
                return pd.Series({
                    "prio_ntl_core_mean": np.nan,
                    "prio_ntl_periphery_mean": np.nan,
                    "prio_ntl_core_periphery_ratio": np.nan,
                    "prio_ntl_cv": np.nan,
                })
            p90 = np.quantile(v, 0.90)
            p50 = np.quantile(v, 0.50)
            core_mean      = v[v >= p90].mean()
            periphery_mean = v[v <= p50].mean()
            mean_v = v.mean()
            return pd.Series({
                "prio_ntl_core_mean":            core_mean,
                "prio_ntl_periphery_mean":        periphery_mean,
                "prio_ntl_core_periphery_ratio":  core_mean / periphery_mean if periphery_mean > 0 else np.nan,
                "prio_ntl_cv":                    v.std() / mean_v if mean_v > 0 else np.nan,
            })

        df_m6 = (
            df_panel.groupby(["iso3", "year"])[ntl_col]
            .apply(_polarization)
            .reset_index()
        )
        print(f"Method 6: {len(df_m6):,} rows × {len(df_m6.columns)} columns")
        print(df_m6.drop(columns=["iso3", "year"]).describe().round(3))
else:
    print("Method 6 skipped — panel not available")

## Method 7 — NTL economic shock index

**What it adds:** A sudden drop in nighttime light intensity at the country level is
a real-time proxy for economic deterioration — sanctions taking effect, conflict
destroying infrastructure, a commodity price crash hitting oil-dependent economies.
Because NTL data is available annually with relatively short lag, this signal is
faster-moving than GDP estimates.

We compute a population-weighted country-year NTL mean and compare it against a
5-year rolling baseline. A z-score below −1.5 is flagged as a shock.

**Features produced:**

| Feature | Description |
|---|---|
| `prio_ntl_popwtd_mean` | Population-weighted mean NTL — better economic proxy than simple mean |
| `prio_ntl_yoy_change` | Year-on-year change in population-weighted NTL |
| `prio_ntl_shock_zscore` | Deviation from 5-year rolling baseline (z-score; negative = decline) |
| `prio_ntl_negative_shock` | 1 if z-score < −1.5 (sharp economic deterioration) |

**Why it matters:** Chen & Nordhaus (2011) validate NTL as a GDP proxy particularly
for countries with poor national accounts. The shock z-score operationalizes the
"economic shock → grievance → mobilization" mechanism without requiring WDI GDP
data, which lags by 18+ months in fragile states.

In [ ]:
df_m7 = pd.DataFrame()

if df_panel is not None and "iso3" in df_panel.columns:
    ntl_col = "nlights_calib_mean"
    pop_col = "pop_gpw_sum"

    if ntl_col not in df_panel.columns or pop_col not in df_panel.columns:
        print("Method 7 skipped — NTL or population column missing")
    else:
        # Population-weighted NTL per country-year
        def _pwm_ntl(g):
            v = g[ntl_col].values.astype(float)
            w = g[pop_col].fillna(0).values.astype(float)
            mask = ~np.isnan(v) & (w > 0)
            if mask.sum() == 0:
                return np.nan
            return np.average(v[mask], weights=w[mask])

        popwtd = (
            df_panel.groupby(["iso3", "year"])
            .apply(_pwm_ntl)
            .rename("prio_ntl_popwtd_mean")
            .reset_index()
        )
        popwtd["year"] = popwtd["year"].astype("Int64")
        popwtd = popwtd.sort_values(["iso3", "year"])

        # Year-on-year change
        popwtd["prio_ntl_yoy_change"] = (
            popwtd.groupby("iso3")["prio_ntl_popwtd_mean"].diff()
        )

        # 5-year rolling baseline (shift 1 to avoid same-year leakage)
        popwtd["_roll5_mean"] = (
            popwtd.groupby("iso3")["prio_ntl_popwtd_mean"]
            .transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean())
        )
        popwtd["_roll5_std"] = (
            popwtd.groupby("iso3")["prio_ntl_popwtd_mean"]
            .transform(lambda s: s.shift(1).rolling(5, min_periods=3).std())
        )

        popwtd["prio_ntl_shock_zscore"] = (
            (popwtd["prio_ntl_popwtd_mean"] - popwtd["_roll5_mean"])
            / popwtd["_roll5_std"].replace(0, np.nan)
        )
        popwtd["prio_ntl_negative_shock"] = (
            popwtd["prio_ntl_shock_zscore"] < -NTL_SHOCK_THRESHOLD
        ).astype("Int8")

        df_m7 = popwtd.drop(columns=["_roll5_mean", "_roll5_std"])

        print(f"Method 7: {len(df_m7):,} rows × {len(df_m7.columns)} columns")
        print(df_m7.drop(columns=["iso3", "year"]).describe().round(3))
else:
    print("Method 7 skipped — panel not available")

## Method 8 — GeoEPR ethnic territory join

**What it adds:** The EPR-Core dataset (notebook 14e) tells us *whether* a country
has politically excluded ethnic groups. GeoEPR extends EPR-Core by mapping each
group's settlement territory as a polygon, enabling us to ask: *what are the
economic conditions inside those territories?*

By intersecting GeoEPR polygons with PRIO-GRID cell centroids and reading off NTL,
population, and travel-time values for each cell, we can compute the economic gap
between the homelands of excluded groups and the homelands of included groups —
a direct spatial operationalization of Cederman, Wimmer & Min's (2010) horizontal
inequality hypothesis.

**Spatial join logic:**
1. Download GeoEPR shapefiles (ETH Zurich ICR group)
2. For each PRIO-GRID cell centroid, find which GeoEPR group polygon contains it
   (point-in-polygon via `geopandas.sjoin`)
3. Each polygon carries a `status` field — classify cells as "excluded territory"
   (Powerless / Discriminated) or "included territory" (Monopoly / Dominant / Partner)
4. Aggregate NTL, population, and travel time to excluded vs. included cells
   per country-year

**Features produced:**

| Feature | Description |
|---|---|
| `prio_geoepr_excl_ntl_mean` | Mean NTL across cells in excluded group territories |
| `prio_geoepr_incl_ntl_mean` | Mean NTL across cells in included group territories |
| `prio_geoepr_ntl_excl_gap` | Normalized NTL gap: (incl − excl) / (incl + excl); range [−1, 1] |
| `prio_geoepr_excl_pop_share` | Excluded territory population / total country population |
| `prio_geoepr_excl_ttime_mean` | Mean travel time to city for excluded territory cells |
| `prio_geoepr_ttime_excl_gap` | Travel-time gap: excluded − included (positive = more remote) |
| `prio_geoepr_n_excl_groups` | Count of distinct excluded groups with mapped territory |

**Why it matters for instability models:**
`prio_geoepr_ntl_excl_gap` is a causal-mechanism variable — it measures not just
whether groups are excluded politically, but whether exclusion translates into
measurable economic deprivation in their physical homelands. Cederman et al. (2011)
show this spatial economic gap is a stronger predictor of ethnic civil war onset
than national-level inequality measures. It is unavailable in any existing
country-year dataset and can only be constructed via this PRIO-GRID × GeoEPR join.

**Requires:** `geopandas` and `shapely`. Install if needed:
```
pip install geopandas shapely
```

In [ ]:
try:
    import geopandas as gpd
    from shapely.geometry import Point
    _GEOPANDAS_OK = True
except ImportError:
    print("geopandas not installed — Method 8 will be skipped.")
    print("  Install with:  pip install geopandas shapely")
    _GEOPANDAS_OK = False

# GeoEPR shapefile ZIP — ETH Zurich ICR group (no registration required)
GEOEPR_URLS = [
    "https://icr.ethz.ch/data/epr/geoepr/GeoEPR-2021.zip",
    "https://icr.ethz.ch/data/epr/geoepr/GeoEPR-2019.zip",
    "https://icr.ethz.ch/data/epr/geoepr/GeoEPR.zip",
]

EXCLUDED_STATUSES = {"powerless", "discriminated"}
INCLUDED_STATUSES = {"monopoly", "dominant", "senior partner", "junior partner"}

In [ ]:
import io, zipfile, tempfile, os as _os

def _load_geoepr(urls: list[str]) -> "gpd.GeoDataFrame | None":
    """Download GeoEPR ZIP and return a GeoDataFrame of group polygons."""
    for url in urls:
        print(f"Trying GeoEPR from {url} ...")
        try:
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
        except Exception as e:
            print(f"  Failed: {e}")
            continue

        try:
            with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                # Write to a temp directory so geopandas can find companion files
                with tempfile.TemporaryDirectory() as tmp:
                    z.extractall(tmp)
                    shp_files = [
                        _os.path.join(root, f)
                        for root, _, files in _os.walk(tmp)
                        for f in files if f.endswith(".shp")
                    ]
                    if not shp_files:
                        print("  No .shp file found in archive")
                        continue
                    gdf = gpd.read_file(shp_files[0])
                    print(f"  Loaded GeoEPR: {len(gdf):,} polygons | cols: {list(gdf.columns)}")
                    return gdf
        except Exception as e:
            print(f"  Parse failed: {e}")
            continue

    print("  Could not load GeoEPR from any URL.")
    return None


gdf_geoepr = None
if _GEOPANDAS_OK:
    import requests as _req
    requests = _req          # make sure requests is available in this scope
    gdf_geoepr = _load_geoepr(GEOEPR_URLS)

    if gdf_geoepr is not None:
        gdf_geoepr.columns = [c.strip().lower() for c in gdf_geoepr.columns]
        # Normalise status field
        for sc in ["status", "gwstatus"]:
            if sc in gdf_geoepr.columns:
                gdf_geoepr["status"] = gdf_geoepr[sc].str.strip().str.lower()
                break
        # Year-range columns
        for fc in ["from", "frm", "styear"]:
            if fc in gdf_geoepr.columns:
                gdf_geoepr["year_from"] = pd.to_numeric(gdf_geoepr[fc], errors="coerce")
                break
        for tc in ["to", "endyear"]:
            if tc in gdf_geoepr.columns:
                gdf_geoepr["year_to"] = pd.to_numeric(gdf_geoepr[tc], errors="coerce")
                break
        # GW country code
        for gc in ["gwid", "gwno", "stateid"]:
            if gc in gdf_geoepr.columns:
                gdf_geoepr["gwno"] = pd.to_numeric(gdf_geoepr[gc], errors="coerce")
                break
        # Group ID for counting distinct excluded groups
        for idc in ["gwgroupid", "groupid", "group_id"]:
            if idc in gdf_geoepr.columns:
                gdf_geoepr["group_id"] = gdf_geoepr[idc]
                break

        print(f"GeoEPR ready: {len(gdf_geoepr):,} polygons, CRS={gdf_geoepr.crs}")

In [ ]:
df_m8 = pd.DataFrame()

if not _GEOPANDAS_OK or gdf_geoepr is None or df_static is None or df_panel is None:
    print("Method 8 skipped — geopandas, GeoEPR, or PRIO-GRID static not available")
else:
    # ── Step 1: Build a GeoDataFrame of PRIO-GRID cell centroids ─────────────
    df_pts = df_static[["gid", "xcoord", "ycoord", "gwno"]].dropna(
        subset=["xcoord", "ycoord"]
    ).copy()
    df_pts["gwno"] = pd.to_numeric(df_pts["gwno"], errors="coerce")

    gdf_cells = gpd.GeoDataFrame(
        df_pts,
        geometry=gpd.points_from_xy(df_pts["xcoord"], df_pts["ycoord"]),
        crs="EPSG:4326",
    )

    # ── Step 2: Ensure CRS match ──────────────────────────────────────────────
    if gdf_geoepr.crs is None:
        gdf_geoepr = gdf_geoepr.set_crs("EPSG:4326")
    elif gdf_geoepr.crs.to_epsg() != 4326:
        gdf_geoepr = gdf_geoepr.to_crs("EPSG:4326")

    # ── Step 3: Spatial join — which polygon does each cell centroid fall in? ─
    # predicate='within' returns one row per cell-polygon match
    # (a cell can fall in multiple overlapping polygons if territories overlap)
    join_cols = ["gid", "status", "year_from", "year_to", "group_id"]
    join_cols = [c for c in join_cols if c in gdf_geoepr.columns or c == "gid"]
    geoepr_for_join = gdf_geoepr[[c for c in ["status", "year_from", "year_to", "group_id", "geometry"]
                                   if c in gdf_geoepr.columns]]

    cell_group = gpd.sjoin(
        gdf_cells[["gid", "gwno", "geometry"]],
        geoepr_for_join,
        how="left",
        predicate="within",
    ).drop(columns=["geometry", "index_right"], errors="ignore")

    # Classify cells
    cell_group["is_excluded"] = cell_group["status"].isin(EXCLUDED_STATUSES).astype(int)
    cell_group["is_included"] = cell_group["status"].isin(INCLUDED_STATUSES).astype(int)

    print(f"Spatial join: {len(cell_group):,} cell-polygon pairs "
          f"({cell_group['gid'].nunique():,} unique cells matched)")
    print(f"  Excluded territory cells: {cell_group['is_excluded'].sum():,}")
    print(f"  Included territory cells: {cell_group['is_included'].sum():,}")

    # ── Step 4: Join time-varying PRIO-GRID data ─────────────────────────────
    # cell_group maps gid → (status, year_from, year_to)
    # We need to expand to cell-year rows, then join NTL/pop from df_panel
    ntl_col   = "nlights_calib_mean"
    pop_col   = "pop_gpw_sum"
    ttime_col = "ttime_mean"
    avail_cols = [c for c in [ntl_col, pop_col, ttime_col] if c in df_panel.columns]

    panel_slim = df_panel[["gid", "year", "iso3"] + avail_cols].copy()

    # Expand cell_group across panel years using year_from/year_to validity window
    cell_years = panel_slim.merge(cell_group[
        ["gid", "status", "is_excluded", "is_included"] +
        (["year_from", "year_to"] if "year_from" in cell_group.columns else []) +
        (["group_id"] if "group_id" in cell_group.columns else [])
    ], on="gid", how="left")

    if "year_from" in cell_years.columns and "year_to" in cell_years.columns:
        yr = cell_years["year"].astype(float)
        cell_years = cell_years[
            (yr >= cell_years["year_from"].fillna(0)) &
            (yr <= cell_years["year_to"].fillna(9999))
        ].copy()

    # ── Step 5: Aggregate to country-year ────────────────────────────────────
    def _grp_mean(df_cy, flag_col, val_col):
        sub = df_cy[df_cy[flag_col] == 1]
        return sub.groupby(["iso3", "year"])[val_col].mean()

    def _grp_sum(df_cy, flag_col, val_col):
        sub = df_cy[df_cy[flag_col] == 1]
        return sub.groupby(["iso3", "year"])[val_col].sum()

    rows = {}

    if ntl_col in cell_years.columns:
        rows["prio_geoepr_excl_ntl_mean"] = _grp_mean(cell_years, "is_excluded", ntl_col)
        rows["prio_geoepr_incl_ntl_mean"] = _grp_mean(cell_years, "is_included", ntl_col)

    if pop_col in cell_years.columns:
        total_pop = cell_years.groupby(["iso3", "year"])[pop_col].sum().replace(0, np.nan)
        excl_pop  = _grp_sum(cell_years, "is_excluded", pop_col)
        rows["prio_geoepr_excl_pop_share"] = (excl_pop / total_pop).clip(0, 1)

    if ttime_col in cell_years.columns:
        rows["prio_geoepr_excl_ttime_mean"] = _grp_mean(cell_years, "is_excluded", ttime_col)
        rows["prio_geoepr_incl_ttime_mean"] = _grp_mean(cell_years, "is_included", ttime_col)

    if "group_id" in cell_years.columns:
        excl_cells = cell_years[cell_years["is_excluded"] == 1]
        rows["prio_geoepr_n_excl_groups"] = (
            excl_cells.groupby(["iso3", "year"])["group_id"].nunique()
        )

    df_m8 = pd.DataFrame(rows).reset_index()

    # Derived gap features
    if "prio_geoepr_excl_ntl_mean" in df_m8.columns and "prio_geoepr_incl_ntl_mean" in df_m8.columns:
        incl = df_m8["prio_geoepr_incl_ntl_mean"]
        excl = df_m8["prio_geoepr_excl_ntl_mean"]
        denom = (incl + excl).replace(0, np.nan)
        df_m8["prio_geoepr_ntl_excl_gap"] = ((incl - excl) / denom).clip(-1, 1)

    if "prio_geoepr_excl_ttime_mean" in df_m8.columns and "prio_geoepr_incl_ttime_mean" in df_m8.columns:
        df_m8["prio_geoepr_ttime_excl_gap"] = (
            df_m8["prio_geoepr_excl_ttime_mean"] - df_m8["prio_geoepr_incl_ttime_mean"]
        )

    print(f"\nMethod 8: {len(df_m8):,} rows × {len(df_m8.columns)} columns")
    print(df_m8.drop(columns=["iso3", "year"]).describe().round(3))

## Combine all method outputs and write to ADLS

Left-join all seven method DataFrames onto the country-year spine so that every
country-year row exists even if individual methods produced no data (e.g., Method 4
only covers country-years with recorded conflict). Missing values are left as NaN —
notebook 14 applies missingness flags during feature matrix construction.

In [ ]:
df_engineered = pd.DataFrame()

if df_panel is not None and "iso3" in df_panel.columns:
    spine = (
        df_panel[["iso3", "year"]]
        .drop_duplicates()
        .sort_values(["iso3", "year"])
        .reset_index(drop=True)
    )

    result = spine.copy()
    for label, df_m in [
        ("M1", df_m1), ("M2", df_m2), ("M3", df_m3),
        ("M4", df_m4), ("M5", df_m5), ("M6", df_m6),
        ("M7", df_m7), ("M8", df_m8),
    ]:
        if df_m is not None and not df_m.empty:
            merge_cols = [c for c in df_m.columns if c not in ("iso3", "year")]
            result = result.merge(df_m[["iso3", "year"] + merge_cols],
                                  on=["iso3", "year"], how="left")
            print(f"  {label}: +{len(merge_cols)} columns  → total {result.shape[1]}")
        else:
            print(f"  {label}: skipped (empty)")

    df_engineered = result
    print(f"\nFinal engineered feature table: {df_engineered.shape}")
    print(f"Feature columns: {[c for c in df_engineered.columns if c not in ('iso3','year')]}")

    write_parquet(
        df_engineered,
        f"raw/prio_grid/{RUN_DATE}/priogrid_engineered_features.parquet",
    )
else:
    print("Nothing to write — panel not loaded")

## Validation — point-biserial correlation with instability labels

Load the labels parquet from notebook 14 (if available) and compute point-biserial
correlations between each engineered feature and each binary outcome label.
High absolute correlations confirm the features carry predictive signal before
they enter the model pipeline.

In [ ]:
from scipy.stats import pointbiserialr

df_labels = read_latest_parquet("processed/feature_matrix", filename_hint="labels")

if df_labels is not None and not df_engineered.empty:
    df_labels.columns = [c.lower().strip() for c in df_labels.columns]
    df_labels["year"]  = pd.to_numeric(df_labels["year"],  errors="coerce").astype("Int64")

    combined = df_engineered.merge(df_labels, on=["iso3", "year"], how="inner")
    print(f"Joined rows for validation: {len(combined):,}")

    outcome_cols = [c for c in df_labels.columns if c not in ("iso3", "year")]
    feature_cols = [c for c in df_engineered.columns if c not in ("iso3", "year")]

    corr_rows = []
    for feat in feature_cols:
        for outcome in outcome_cols:
            x = combined[feat]
            y = combined[outcome]
            mask = x.notna() & y.notna() & y.isin([0, 1])
            if mask.sum() < 30:
                continue
            r, p = pointbiserialr(x[mask].astype(float), y[mask].astype(int))
            corr_rows.append({"feature": feat, "outcome": outcome, "r": round(r, 4), "p": round(p, 4)})

    df_corr = pd.DataFrame(corr_rows).sort_values("r", key=abs, ascending=False)

    print("\nTop 20 feature–outcome correlations (by |r|):")
    print(df_corr.head(20).to_string(index=False))

    # Pivot: features × outcomes heatmap-ready table
    corr_pivot = df_corr.pivot(index="feature", columns="outcome", values="r")
    print("\nCorrelation matrix (features × outcomes):")
    print(corr_pivot.round(3).to_string())
else:
    print("Validation skipped — labels parquet not available or engineered features empty")

## Summary

In [ ]:
print("=" * 60)
print("PRIO-GRID feature engineering complete")
print("=" * 60)

method_results = [
    ("1 — Distributional aggregates",           df_m1),
    ("2 — NTL Gini / darkness fraction",         df_m2),
    ("3 — Pop-weighted climate stress",          df_m3),
    ("4 — Conflict cell density (UCDP join)",    df_m4),
    ("5 — Lootable resource pop exposure",       df_m5),
    ("6 — Economic core/periphery polarization", df_m6),
    ("7 — NTL economic shock index",             df_m7),
    ("8 — GeoEPR ethnic territory join",         df_m8),
]

total_features = 0
for name, df_m in method_results:
    if df_m is not None and not df_m.empty:
        n_feat = len([c for c in df_m.columns if c not in ("iso3", "year")])
        total_features += n_feat
        print(f"  Method {name}: {n_feat} features")
    else:
        print(f"  Method {name}: SKIPPED")

print(f"\n  Total new features : {total_features}")
if not df_engineered.empty:
    print(f"  Output rows        : {len(df_engineered):,} country-years")
print()
print("ADLS path written:")
print(f"  raw/prio_grid/{RUN_DATE}/priogrid_engineered_features.parquet")
print()
print("Next step: add 'priogrid_engineered' to RAW_PREFIXES in notebook 14")